In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
hh = pd.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\tdm_market_analysis\hh.csv")
hh

In [ ]:
hh.columns

### vehicle

In [ ]:
hh[["hhsize", "hhvehs"]].describe()

In [ ]:
sf_hh = hh[hh["hhtaz"] < 1000].copy()
sf_hh

In [ ]:
auto_ownership = (
    sf_hh
    .groupby(["hhsize", "hhvehs"], as_index=False)
    .agg(
        unweighted=("hhno", "size"),      # sample households
        weighted=("hhexpfac", "sum")      # estimated actual households
    )
)

auto_ownership

In [ ]:
auto_ownership["weighted"].sum()

In [ ]:
auto_ownership.to_csv(r"./auto_ownership.csv", index=False)

### tour rates

In [ ]:
tour = pd.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\tdm_market_analysis\tour.csv")
tour

In [ ]:
tour.columns

In [ ]:
tour["pdpurp"].unique()

In [ ]:
person = pd.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\tdm_market_analysis\person.csv")
person

In [ ]:
person.columns

In [ ]:
sf_tour = tour[tour["hhtaz"] < 1000].copy()
sf_tour

In [ ]:
sf_tour.columns

In [ ]:
sf_tour["tour_type"] = np.where(
    sf_tour["parent"] == 0,
    "Home-based",
    "Non-home-based"
)

# Check
sf_tour["tour_type"].value_counts()

In [ ]:
# ------------------------------------------------------------
# Weighted tours by purpose and mode
# ------------------------------------------------------------

tour_rate = (
    sf_tour
    .groupby(["pdpurp", "tmodetp", "tour_type"], as_index=False)
    .agg(
        unweighted_tours=("tour", "size"),
        weighted_tours=("toexpfac", "sum")
    )
)

# ------------------------------------------------------------
# Weighted number of persons
# One person may have multiple tours, so deduplicate first
# ------------------------------------------------------------

weighted_persons = (
    person.loc[person["hhtaz"] < 1000]
    .drop_duplicates(["hhno", "pno"])
    ["psexpfac"]
    .sum()
)

# ------------------------------------------------------------
# Tour rate per person
# ------------------------------------------------------------

tour_rate["tours_per_person"] = (
    tour_rate["weighted_tours"]
    / weighted_persons
)

tour_rate

In [ ]:
weighted_persons

In [ ]:
sf_tour.loc[(sf_tour["pdpurp"] == 2) & (sf_tour["tmodetp"] == 3) & (sf_tour["tour_type"] == "Non-home-based"), "toexpfac"].sum()

In [ ]:
tour_rate.to_csv(r"./tour_rate.csv", index=False)

### trip rates

In [ ]:
trip = pd.read_csv(r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\tdm_market_analysis\trip.csv")
trip

In [ ]:
trip.columns

In [ ]:
trip["dpurp"].unique()

In [ ]:
trip["mode"].unique()

In [ ]:
trip = trip.merge(person[["hhno", "pno", "hhtaz"]], on=["hhno", "pno"], how="left")
trip

In [ ]:
sf_trip = trip[trip["hhtaz"] < 1000].copy()
sf_trip

In [ ]:
sf_trip.columns

In [ ]:
# ------------------------------------------------------------
# Weighted trips by purpose and mode
# ------------------------------------------------------------

trip_rate = (
    sf_trip
    .groupby(["dpurp", "mode"], as_index=False)
    .agg(
        unweighted_trips=("trexpfac", "size"),
        weighted_trips=("trexpfac", "sum")
    )
)

# ------------------------------------------------------------
# Weighted number of persons
# One person may have multiple trips, so deduplicate first
# ------------------------------------------------------------

weighted_persons = (
    person.loc[person["hhtaz"] < 1000]
    .drop_duplicates(["hhno", "pno"])
    ["psexpfac"]
    .sum()
)

# ------------------------------------------------------------
# Trip rate per person
# ------------------------------------------------------------

trip_rate["trips_per_person"] = (
    trip_rate["weighted_trips"]
    / weighted_persons
)

trip_rate

In [ ]:
weighted_persons

In [ ]:
trip_rate.to_csv(r"./trip_rate.csv", index=False)

### trip dest activity durations by purpose

In [ ]:
sf_trip

In [ ]:
# Keep original columns
sf_trip[["arrtm", "endacttm"]].head()

In [ ]:
print(sf_trip["arrtm"].describe())
print(sf_trip["endacttm"].describe())

print(sf_trip["arrtm"].sort_values().unique()[:20])
print(sf_trip["arrtm"].sort_values().unique()[-20:])

print(sf_trip["endacttm"].sort_values().unique()[:20])
print(sf_trip["endacttm"].sort_values().unique()[-20:])

In [ ]:
def hhmm_to_minutes(x):
    if pd.isna(x):
        return np.nan

    x = int(x)

    hour = x // 100
    minute = x % 100

    # Handle 2400 as midnight
    if hour == 24 and minute == 0:
        return 24 * 60

    # Invalid HHMM
    if hour > 23 or minute > 59:
        return np.nan

    return hour * 60 + minute

In [ ]:
sf_trip["arr_minutes"] = (
    sf_trip["arrtm"]
    .apply(hhmm_to_minutes)
)

sf_trip["endact_minutes"] = (
    sf_trip["endacttm"]
    .apply(hhmm_to_minutes)
)

In [ ]:
sf_trip[["arr_minutes", "endact_minutes"]].head()

In [ ]:
sf_trip["activity_duration_min"] = (
    sf_trip["endact_minutes"]
    - sf_trip["arr_minutes"]
)

sf_trip[["activity_duration_min"]].head()

In [ ]:
overnight = sf_trip["activity_duration_min"] < 0

sf_trip.loc[
    overnight,
    "activity_duration_min"
] += 24 * 60

In [ ]:
sf_trip["activity_duration_hr"] = (
    sf_trip["activity_duration_min"] / 60
)

In [ ]:
sf_trip[
    ["hhno", "pno", "day", "tour", "half", "arrtm", "endacttm",
     "arr_minutes", "endact_minutes",
     "activity_duration_min", "activity_duration_hr"]
].head(20)

In [ ]:
# ------------------------------------------------------------
# Weighted average activity duration by purpose
# ------------------------------------------------------------

sf_trip["weighted_duration_hr"] = (
    sf_trip["activity_duration_hr"] * sf_trip["trexpfac"]
)

duration_summary = (
    sf_trip
    .groupby("dpurp", as_index=False)
    .agg(
        unweighted_trips=("trexpfac", "size"),
        weighted_trips=("trexpfac", "sum"),
        weighted_duration_hr=("weighted_duration_hr", "sum")
    )
)

duration_summary["avg_duration_hr"] = (
    duration_summary["weighted_duration_hr"]
    / duration_summary["weighted_trips"]
)

duration_summary

In [ ]:
duration_summary.to_csv(r"./trip_activity_duration_summary.csv", index=False)

In [ ]:
# Exploratory 30-minute bins
bins = np.arange(0, 24.5, 0.5)

sf_trip["duration_bin_30min"] = pd.cut(
    sf_trip["activity_duration_hr"],
    bins=bins,
    right=False,
    include_lowest=True
)

duration_dist = (
    sf_trip
    .groupby(
        ["dpurp", "duration_bin_30min"],
        observed=True,
        as_index=False
    )
    .agg(
        unweighted_trips=("trexpfac", "size"),
        weighted_trips=("trexpfac", "sum")
    )
)

# Percentage within each purpose
duration_dist["weighted_pct"] = (
    duration_dist["weighted_trips"]
    / duration_dist.groupby("dpurp")["weighted_trips"].transform("sum")
    * 100
)

duration_dist

In [ ]:
duration_pivot = (
    duration_dist
    .pivot(
        index="dpurp",
        columns="duration_bin_30min",
        values=[
            "unweighted_trips",
            "weighted_trips",
            "weighted_pct"
        ]
    )
)

duration_pivot

In [ ]:
duration_pivot.to_csv(r"./trip_activity_duration_pivot.csv", index=False)

In [ ]:
for purpose in sorted(sf_trip["dpurp"].dropna().unique()):

    temp = duration_dist[
        duration_dist["dpurp"] == purpose
    ]

    plt.figure(figsize=(10, 4))

    plt.bar(
        temp["duration_bin_30min"].astype(str),
        temp["weighted_pct"]
    )

    plt.title(f"Destination Activity Duration - Purpose {purpose}")
    plt.xlabel("Duration (hours)")
    plt.ylabel("Weighted percent of trips")

    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

### trip start times by dest purpose by hour of day

In [ ]:
sf_trip

In [ ]:
# ------------------------------------------------------------
# 1. Clean departure time
# ------------------------------------------------------------

sf_trip["dept_minutes"] = (
    sf_trip["deptm"]
    .apply(hhmm_to_minutes)
)

# Check invalid / missing times created during cleaning
sf_trip.loc[
    sf_trip["dept_minutes"].isna(),
    ["deptm"]
].value_counts()

In [ ]:
# ------------------------------------------------------------
# 2. Create hour-of-day variable
# ------------------------------------------------------------

sf_trip["dept_hour"] = (
    (sf_trip["dept_minutes"] // 60) % 24
).astype("Int64")

In [ ]:
sf_trip[
    ["deptm", "dept_minutes", "dept_hour"]
].head(20)

In [ ]:
sf_trip["dept_hour"].value_counts().sort_index()

In [ ]:
# ------------------------------------------------------------
# 3. Estimated trip starts by destination purpose and hour
# ------------------------------------------------------------

trip_start_time = (
    sf_trip
    .dropna(subset=["dpurp", "dept_hour"])
    .groupby(
        ["dpurp", "dept_hour"],
        as_index=False
    )
    .agg(
        unweighted_trips=("trexpfac", "size"),
        weighted_trips=("trexpfac", "sum")
    )
)

trip_start_time["weighted_pct"] = (
    trip_start_time["weighted_trips"]
    / trip_start_time
        .groupby("dpurp")["weighted_trips"]
        .transform("sum")
    * 100
)

trip_start_time

In [ ]:
trip_start_time.to_csv(r"./trip_start_time.csv", index=False)

In [ ]:
trip_start_pivot = (
    trip_start_time
    .pivot(
        index="dpurp",
        columns="dept_hour",
        values=[
            "unweighted_trips",
            "weighted_trips",
            "weighted_pct"
        ]
    )
    .reset_index()
)

trip_start_pivot

In [ ]:
trip_start_pivot.to_csv(r"./trip_start_time_pivot.csv", index=False)

In [ ]:
for purpose in sorted(trip_start_time["dpurp"].unique()):

    temp = (
        trip_start_time[
            trip_start_time["dpurp"] == purpose
        ]
        .sort_values("dept_hour")
    )

    plt.figure(figsize=(10, 4))

    plt.bar(
        temp["dept_hour"],
        temp["weighted_pct"],
        width=0.9
    )

    plt.title(
        f"Trip Start Time Distribution - Destination Purpose {purpose}"
    )
    plt.xlabel("Hour of Day")
    plt.ylabel("Weighted Percent of Trips (%)")

    plt.xticks(range(24))
    plt.xlim(-0.5, 23.5)

    plt.tight_layout()
    plt.show()

### if trips include sub-tour?

In [ ]:
sf_trip

In [ ]:
sf_trip.columns

In [ ]:
# Sort trips in sequence within each person's tour
sf_trip_check = (
    sf_trip
    .sort_values(["hhno", "pno", "tour", "tsvid"])
    .copy()
)

# Get the next trip's departure time within the same tour
sf_trip_check["next_deptm"] = (
    sf_trip_check
    .groupby(["hhno", "pno", "tour"])["deptm"]
    .shift(-1)
)

# Optional: also show the next trip's tsvid
sf_trip_check["next_tsvid"] = (
    sf_trip_check
    .groupby(["hhno", "pno", "tour"])["tsvid"]
    .shift(-1)
)

# Keep cases where:
# 1. there IS a next trip within the same tour
# 2. current endacttm != next trip's deptm
mismatch = sf_trip_check[
    sf_trip_check["next_deptm"].notna()
    & (sf_trip_check["endacttm"] != sf_trip_check["next_deptm"])
].copy()

In [ ]:
mismatch[
    [
        "hhno",
        "pno",
        "day",
        "tour",
        "tsvid",
        "tseg",
        "deptm",
        "arrtm",
        "endacttm",
        "next_tsvid",
        "next_deptm"
    ]
]